# UWCT Model A tuning sprint, round 1 (D-038): E011-E019

**Plan (D-038):** yolo26s / m / l x seeds 0 / 1 / 2, epochs 60 as a ceiling with patience 10
(a run trains while val mAP50-95 improves and stops 10 epochs after its last improvement), dataset v4 (frozen), val = 324 v1 frames.
The setup cells are the verified E007 cells (archive sha256, raw checks, v4 build, Ultralytics VERIFIED).
New in this notebook:

- the repo is cloned straight from `p4/colab-baseline` (E011-E019 configs are merged there);
- **Cell 11** smoke-tests all three model sizes, two at a time (one per T4);
- **Cell 12** trains the queue on both GPUs at once, seed 0 of every size first (E011 s, E013 l, E012 m). A run starts only if it can finish before
  9.5 h after Cell 1 ran (worst case: all 60 epochs); every finished run is zipped immediately (`/kaggle/working/E0xx_package.zip`).
- **Selection metric (D-038): val mAP50-95**, the same metric Ultralytics uses to pick `best.pt`; mAP50, P and R are reported next to it.

**How to run:** run Cells 1-11 interactively and check the smoke output, stop the session, then
*Save Version -> Save & Run All* (accelerator **GPU T4 x2**, internet on, secret `GITHUB_TOKEN` attached).
The leakage check (E007 notebook Cell 12) is not repeated: v4 is frozen and passed it on 2026-11-03 and 2026-11-08.

In [ ]:

# Cell 1 - inspect the environment (read-only)
# 1) Where Kaggle mounted our 4 datasets, and the files inside each
# 2) Which GPU we got and how much memory it has
# 3) How much free disk space /kaggle/working and /tmp have (we must unzip ~17 GB)

import os, subprocess, time

# notebook start time: Cell 12's deadline (D-038 point 2) is measured from here
os.makedirs('/kaggle/working/uwct_logs', exist_ok=True)
with open('/kaggle/working/uwct_logs/t0.txt', 'w') as fh:
    fh.write(str(time.time()))
print('run type:', os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'unknown'))

print("=== INPUTS ===")
print(subprocess.run("ls -la /kaggle/input/*", shell=True, capture_output=True, text=True).stdout)

print("=== GPU ===")
print(subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True).stdout)

print("=== DISK ===")
print(subprocess.run("df -h /kaggle/working /tmp", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# Cell 2 - inspect the attached datasets (read-only)
# For each dataset mounted under /kaggle/input/<owner>/<dataset>/ we print
# at most 15 entries, so the output stays small even if a dataset has
# thousands of files. We also count the files in each one.

import os

INPUT_ROOT = "/kaggle/input"

for owner in sorted(os.listdir(INPUT_ROOT)):
    owner_dir = os.path.join(INPUT_ROOT, owner)
    for ds in sorted(os.listdir(owner_dir)):
        ds_dir = os.path.join(owner_dir, ds)
        n_files = sum(len(files) for _, _, files in os.walk(ds_dir))
        print(f"=== {owner}/{ds}   ({n_files} files in total)")
        for i, name in enumerate(sorted(os.listdir(ds_dir))):
            if i >= 15:
                print("    ... (more entries not shown)")
                break
            path = os.path.join(ds_dir, name)
            if os.path.isfile(path):
                print(f"    {name}   {os.path.getsize(path):,} bytes")
            else:
                print(f"    {name}/")
        print()

In [ ]:
# Cell 3 - locate our 4 uploaded archives and check their sizes (read-only)
# Expected sizes are the byte counts of the verified local copies (2026-11-06).
# A size match is a quick first check; the sha256 check comes in the next cell.

import os

OUR_ROOT = "/kaggle/input/datasets/sakshampainuly007"
EXPECTED_BYTES = {
    "biology.zip.bin":                5_478_527_588,
    "harsh_model_a_2026-11.zip.bin": 10_121_406_479,
    "harsh_v1_extra_2026-11.zip.bin":  1_819_177_935,
    "harsh_v1_colour_2026-11.zip.bin":   263_128_178,
}

found = {}
for dirpath, _, files in os.walk(OUR_ROOT):
    for f in files:
        found[f] = os.path.join(dirpath, f)

for name, expected in EXPECTED_BYTES.items():
    if name not in found:
        print(f"MISSING   {name}")
        continue
    size = os.path.getsize(found[name])
    status = "OK      " if size == expected else "MISMATCH"
    print(f"{status}  {found[name]}")
    print(f"          size {size:,} B (expected {expected:,} B)")

extra = set(found) - set(EXPECTED_BYTES)
if extra:
    print("Unexpected files:", sorted(extra))

In [ ]:
# Cell 4 - verify archive integrity with sha256
# Expected hashes were computed on the verified local copies before upload
# (D:\uwct_kaggle_upload\*\*.sha256). All 4 must print MATCH before we unpack.
# The result is also saved to /kaggle/working/uwct_logs/archive_sha256.txt,
# which is kept when the notebook version is saved.

import hashlib, os, time

ARCHIVES = {
    "/kaggle/input/datasets/sakshampainuly007/uwct-biology-raw/biology.zip.bin":
        "88d3ba76e3e9ed55af7929667e8e3abbfb51f6d7444130c27627cbc3494899b3",
    "/kaggle/input/datasets/sakshampainuly007/uwct-harsh-model-a-2026-11/harsh_model_a_2026-11.zip.bin":
        "ecbcb2a58403365b79c6764d714d414ca97a0e4c80d088db43518f76b76dc2c3",
    "/kaggle/input/datasets/sakshampainuly007/uwct-harsh-v1-extra-2026-11/harsh_v1_extra_2026-11.zip.bin":
        "02ae7b10f23a6f5c42b8598039b3a74c7e0ded47d3bcea5d124f1b3e4e606cbd",
    "/kaggle/input/datasets/sakshampainuly007/uwct-harsh-v1-colour-2026-11/harsh_v1_colour_2026-11.zip.bin":
        "405c373d1d3680b98433ab537c83e73fd8a961a607ae616415d6cfaf2c1852e7",
}

LOG_DIR = "/kaggle/working/uwct_logs"
os.makedirs(LOG_DIR, exist_ok=True)

def sha256_of(path, chunk=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

lines, all_ok = [], True
for path, expected in ARCHIVES.items():
    t0 = time.time()
    got = sha256_of(path)
    ok = got == expected
    all_ok &= ok
    line = f"{'MATCH   ' if ok else 'MISMATCH'}  {os.path.basename(path)}  {got}  ({time.time() - t0:.0f} s)"
    print(line)
    lines.append(line)

summary = "ALL 4 ARCHIVES VERIFIED" if all_ok else "STOP: at least one archive does not match"
print("\n" + summary)
with open(os.path.join(LOG_DIR, "archive_sha256.txt"), "w") as f:
    f.write("\n".join(lines + ["", summary]) + "\n")

In [ ]:
# Cell 5 - unpack the verified archives to /tmp/uwct_raw (raw data, treated read-only)
# /tmp is used because /kaggle/working is only 20 GB; /tmp is wiped when the
# session ends, so this cell must be re-run in every new session.
# Each archive has one top folder named like our repo's data/raw/<folder>.
# Safety: every entry is checked BEFORE extraction (no absolute paths, no "..",
# correct top folder). Nothing is overwritten: an existing target stops the cell.

import os, zipfile, shutil, time

RAW_DIR = "/tmp/uwct_raw"
IN = "/kaggle/input/datasets/sakshampainuly007"
ARCHIVES = {   # archive path -> expected top folder
    f"{IN}/uwct-biology-raw/biology.zip.bin":                              "biology",
    f"{IN}/uwct-harsh-model-a-2026-11/harsh_model_a_2026-11.zip.bin":      "harsh_model_a_2026-11",
    f"{IN}/uwct-harsh-v1-extra-2026-11/harsh_v1_extra_2026-11.zip.bin":    "harsh_v1_extra_2026-11",
    f"{IN}/uwct-harsh-v1-colour-2026-11/harsh_v1_colour_2026-11.zip.bin":  "harsh_v1_colour_2026-11",
}

os.makedirs(RAW_DIR, exist_ok=True)
log = []
for archive, top in ARCHIVES.items():
    target = os.path.join(RAW_DIR, top)
    if os.path.exists(target):
        raise SystemExit(f"STOP: {target} already exists - not overwriting. "
                         f"Restart the session or remove it deliberately.")
    t0 = time.time()
    with zipfile.ZipFile(archive) as zf:
        members = zf.infolist()
        bad = [m.filename for m in members
               if m.filename.startswith("/") or ".." in m.filename.split("/")
               or m.filename.split("/")[0] != top]
        if bad:
            raise SystemExit(f"STOP: unsafe or unexpected entries in {archive}: {bad[:5]}")
        zf.extractall(RAW_DIR)
    n_files = sum(1 for m in members if not m.is_dir())
    line = f"{top:26s} {n_files:6d} files extracted   ({time.time() - t0:.0f} s)"
    print(line)
    log.append(line)

total, used, free = shutil.disk_usage(RAW_DIR)
disk = f"/tmp free after extraction: {free / 1e9:.0f} GB"
print("\n" + disk)
with open("/kaggle/working/uwct_logs/extraction.txt", "w") as f:
    f.write("\n".join(log + ["", disk]) + "\n")

In [ ]:
# Cell 6 - clone the project repository (private) at the integration branch p4/colab-baseline
# The token is read from Kaggle Secrets and passed as a one-off HTTP header,
# so it is NOT written into the repo's .git/config and never printed.
# The exact commit used is printed and logged, so every result in this
# notebook can be traced back to the code that produced it.

import base64, os, subprocess
from kaggle_secrets import UserSecretsClient

REPO_URL = "https://github.com/SakshamPainuly/underwater-creature-tracking.git"
BRANCH = "p4/colab-baseline"
REPO_DIR = "/tmp/uwct_repo"

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()

if os.path.exists(REPO_DIR):
    raise SystemExit(f"STOP: {REPO_DIR} already exists - not overwriting.")

r = subprocess.run(
    ["git", "-c", f"http.extraHeader=Authorization: Basic {auth}",
     "clone", "--quiet", "--branch", BRANCH, REPO_URL, REPO_DIR],
    capture_output=True, text=True)
del token, auth
if r.returncode != 0:
    # print git's message with any secret material removed
    raise SystemExit("STOP: clone failed:\n" + r.stderr.replace(REPO_URL, "<repo>"))

def git(*args):
    return subprocess.run(["git", "-C", REPO_DIR, *args],
                          capture_output=True, text=True).stdout.strip()

info = [
    f"repo    : {REPO_URL}",
    f"branch  : {git('rev-parse', '--abbrev-ref', 'HEAD')}",
    f"commit  : {git('rev-parse', 'HEAD')}",
    f"message : {git('log', '-1', '--format=%s (%ad)', '--date=short')}",
    f"dirty   : {'yes' if git('status', '--porcelain') else 'no'}",
]
print("\n".join(info))
with open("/kaggle/working/uwct_logs/code_version.txt", "w") as f:
    f.write("\n".join(info) + "\n")

In [ ]:
# Cell 7 - raw data check (CLAUDE.md section 4): extracted files vs committed fingerprints
#   biology : structural fingerprint (D-011)  -> paths + sizes (+ txt/xlsx/docx hashes)
#   Harsh x3: full fingerprints (D-031)        -> sha256 of every file (--full)
# Every check must end with "RAW DATA MATCHES FINGERPRINT".

import subprocess, sys

REPO = "/tmp/uwct_repo"
RAW = "/tmp/uwct_raw"
FP = f"{REPO}/data/processed"
CHECKS = [
    ("biology",                 f"{FP}/raw_fingerprint_d011_post_cleanup.json",       []),
    ("harsh_model_a_2026-11",   f"{FP}/raw_fingerprint_harsh_model_a_2026-11.json",   ["--full"]),
    ("harsh_v1_extra_2026-11",  f"{FP}/raw_fingerprint_harsh_v1_extra_2026-11.json",  ["--full"]),
    ("harsh_v1_colour_2026-11", f"{FP}/raw_fingerprint_harsh_v1_colour_2026-11.json", ["--full"]),
]

report, all_ok = [], True
for folder, fingerprint, extra in CHECKS:
    r = subprocess.run([sys.executable, f"{REPO}/scripts/check_raw_unchanged.py",
                        "--root", f"{RAW}/{folder}", "--fingerprint", fingerprint, *extra],
                       capture_output=True, text=True)
    all_ok &= r.returncode == 0
    block = f"===== {folder}\n{r.stdout.strip()}" + (f"\n{r.stderr.strip()}" if r.stderr.strip() else "")
    print(block + "\n")
    report.append(block)

summary = "ALL 4 RAW CHECKS PASSED" if all_ok else "STOP: a raw check failed"
print(summary)
with open("/kaggle/working/uwct_logs/raw_check.txt", "w") as f:
    f.write("\n\n".join(report) + f"\n\n{summary}\n")


In [ ]:
# Cell 8 - install the pinned training package and record the environment (CLAUDE.md section 6)
# requirements-gpu.txt pins ultralytics==8.4.155 and says: do not reinstall torch.
# We print torch's version before and after to prove the install left it untouched.
# The environment report is saved to /kaggle/working/uwct_logs/env_report.json.

import subprocess, sys, json

REPO = "/tmp/uwct_repo"
torch_ver = lambda: subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"],
                                   capture_output=True, text=True).stdout.strip()

before = torch_ver()
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements-gpu.txt"],
                   capture_output=True, text=True)
print("pip exit code:", r.returncode)
tail = (r.stdout + r.stderr).strip().splitlines()[-5:]
if tail:
    print("pip (last lines):\n  " + "\n  ".join(tail))
after = torch_ver()
print(f"torch before: {before}   after: {after}   {'UNCHANGED' if before == after else 'CHANGED - tell Claude'}")

r = subprocess.run([sys.executable, f"{REPO}/scripts/env_report.py",
                    "--out", "/kaggle/working/uwct_logs/env_report.json"],
                   capture_output=True, text=True)
print(r.stderr.strip() or "")
print(json.dumps(json.load(open("/kaggle/working/uwct_logs/env_report.json")), indent=1))

In [ ]:
# Cell 9 - build the YOLO view of frozen dataset v4 (D-032)
# Expected time: about 5-10 min.
# Raw roots point at the verified extracted data in /tmp/uwct_raw (read-only by the builder).
# Output goes to /tmp (not /kaggle/working: ~6 GB of image copies, rebuilt each session).
# The fingerprint must equal the one committed in the E007 config.

import json, os, subprocess, sys, time

REPO = "/tmp/uwct_repo"
RAW = "/tmp/uwct_raw"
OUT = "/tmp/uwct_yolo/lobster_mixed_v4"
EXPECTED_FP = "bbd4a8b172fd590b18d2e8c26be05c85cb3c3246a2e60a3b3513590b931cd13f"

env = dict(os.environ,
           UWCT_RAW_ROOT=f"{RAW}/biology",
           UWCT_HARSH_MODEL_A_ROOT=f"{RAW}/harsh_model_a_2026-11",
           UWCT_HARSH_V1_EXTRA_ROOT=f"{RAW}/harsh_v1_extra_2026-11",
           UWCT_HARSH_V1_COLOUR_ROOT=f"{RAW}/harsh_v1_colour_2026-11")
t0 = time.time()
r = subprocess.run([sys.executable, f"{REPO}/scripts/prepare_yolo_dataset.py", "--version", "v4", "--out", OUT],
                   capture_output=True, text=True, env=env)
if r.returncode != 0:
    raise SystemExit("STOP: build failed:\n" + (r.stdout + r.stderr)[-3000:])

s = json.load(open(f"{OUT}/build_summary.json"))
print(f"built in {time.time() - t0:.0f} s   version {s['dataset_version']} ({s['version_status']})   "
      f"git {s['git']['commit'][:7]} dirty={s['git']['dirty']}")
for split, c in {**s["splits"], **s["eval_only_splits"]}.items():
    print(f"  {split:15s} images {c['images']:5d}   with lobster {c['positive']:5d}   "
          f"empty {c['negative']:4d}   boxes {c['instances']:6d}")
print("  not built:", sum(s["not_included"].values()), s["not_included"])
ok = s["dataset_fingerprint"] == EXPECTED_FP
print(f"\nfingerprint {s['dataset_fingerprint']}\n{'MATCHES the E007 config' if ok else 'STOP: differs from the E007 config'}")

os.system(f"cp {OUT}/build_summary.json /kaggle/working/uwct_logs/build_summary_v4.json")

In [ ]:
# Cell 10 - Ultralytics reads the built dataset (CLAUDE.md section 6: must print VERIFIED)
# Expected time: about 1-2 min (Ultralytics scans ~5,200 images and labels once).
# Checks per split that Ultralytics sees the same images / empty frames / boxes as build_summary.json,
# including the held-out Zenodo sk split (its own data file).

import subprocess, sys, time

REPO = "/tmp/uwct_repo"
OUT = "/tmp/uwct_yolo/lobster_mixed_v4"

t0 = time.time()
r = subprocess.run([sys.executable, f"{REPO}/scripts/verify_yolo_dataset.py", "--dataset", OUT],
                   capture_output=True, text=True)
print(r.stdout.strip())
print(f"\n(exit code {r.returncode}, {time.time() - t0:.0f} s)")
if r.returncode != 0:
    print("stderr (last lines):\n" + "\n".join(r.stderr.strip().splitlines()[-15:]))
with open("/kaggle/working/uwct_logs/verify_yolo_v4.txt", "w") as f:
    f.write(r.stdout)

In [ ]:
# Cell 11 - smoke tests (CLAUDE.md section 6) for all three model sizes, two at a time
# Expected time: about 8-15 min (includes downloading yolo26s/m/l.pt).
# GPU 0: E011 (yolo26s) then E012 (yolo26m); GPU 1: E013 (yolo26l). Each is 1 epoch on 10% of train,
# validated on val, with every training-side check (config, clean commit, frozen v4 view, checkpoint).
# Prints per model: exit code, seconds for the 10% epoch, peak GPU memory. NOT results.
# It also measures how much slower a run gets when two share the 4 CPUs.

import csv, glob, json, os, re, signal, subprocess, sys, threading, time

REPO = "/tmp/uwct_repo"
RAW = "/tmp/uwct_raw"
DATASET = "/tmp/uwct_yolo/lobster_mixed_v4"
W = "/kaggle/working"
RUNS = f"{W}/uwct_runs"           # kept when the notebook version is saved
WEIGHTS = f"{W}/uwct_weights"     # pretrained checkpoint downloads
LOGS = f"{W}/uwct_logs"
CFG = f"{REPO}/configs/experiments"
os.makedirs(RUNS, exist_ok=True); os.makedirs(WEIGHTS, exist_ok=True)

ENV = dict(os.environ,
           UWCT_RAW_ROOT=f"{RAW}/biology",
           UWCT_HARSH_MODEL_A_ROOT=f"{RAW}/harsh_model_a_2026-11",
           UWCT_HARSH_V1_EXTRA_ROOT=f"{RAW}/harsh_v1_extra_2026-11",
           UWCT_HARSH_V1_COLOUR_ROOT=f"{RAW}/harsh_v1_colour_2026-11")

N_GPUS = len([l for l in subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines()
              if l.startswith("GPU ")])
PRINT_LOCK = threading.Lock()

def say(*a):
    with PRINT_LOCK:
        print(*a, flush=True)

def launch(config_name, gpu, log_path, extra=()):
    """One train_yolo.py process that sees only GPU `gpu` (it becomes its cuda:0)."""
    env = dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu))
    cmd = [sys.executable, f"{REPO}/scripts/train_yolo.py", "--config", f"{CFG}/{config_name}",
           "--runs-root", RUNS, "--weights-dir", WEIGHTS, "--dataset", DATASET, *extra]
    log = open(log_path, "w")
    return subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, text=True, env=env,
                            start_new_session=True), log

def peak_gpu_gb(log_path):
    vals = [float(x) for x in re.findall(r"\s(\d+(?:\.\d+)?)G\s", open(log_path, errors="replace").read())]
    return max(vals) if vals else float("nan")

def rows_of(results_csv):
    if not os.path.exists(results_csv):
        return []
    with open(results_csv) as fh:
        return [{k.strip(): v for k, v in r.items()} for r in csv.DictReader(fh)]

SMOKES = {0: ["E011_yolos_ep60_seed0.yaml", "E012_yolom_ep60_seed0.yaml"],
          1: ["E013_yolol_ep60_seed0.yaml"]}
if N_GPUS < 2:
    SMOKES = {0: SMOKES[0] + SMOKES[1]}
    print("Only one GPU visible: smoke tests run one after another (check the accelerator is GPU T4 x2).")

smoke_result = {}
def smoke_lane(gpu, configs):
    for name in configs:
        exp = name.split("_")[0]
        log_path = f"{LOGS}/smoke_{exp}.log"
        t0 = time.time()
        proc, log = launch(name, gpu, log_path, ["--smoke"])
        rc = proc.wait(); log.close()
        res = {"exit": rc, "minutes": round((time.time() - t0) / 60, 1), "peak_gpu_gb": peak_gpu_gb(log_path)}
        if rc == 0:
            run_dir = sorted(glob.glob(f"{RUNS}/{exp}_smoke_*"))[-1]
            rows = rows_of(f"{run_dir}/train/results.csv")
            res["epoch_s_10pct"] = round(float(rows[-1]["time"])) if rows else None
            res["checkpoint_sha256"] = json.load(open(f"{run_dir}/run_record.json"))["model"]["checkpoint_sha256"][:12]
        else:
            res["log_tail"] = open(log_path, errors="replace").read().splitlines()[-20:]
        smoke_result[exp] = res
        say(f"GPU {gpu}  {exp}: {res}")

threads = [threading.Thread(target=smoke_lane, args=(g, c)) for g, c in SMOKES.items()]
[t.start() for t in threads]; [t.join() for t in threads]

json.dump(smoke_result, open(f"{LOGS}/smoke_summary.json", "w"), indent=1)
ok = all(r["exit"] == 0 for r in smoke_result.values()) and len(smoke_result) == 3
print("\nALL 3 SMOKE TESTS PASSED" if ok else "\nSTOP: a smoke test failed - show Claude the output above")
if not ok:
    raise SystemExit(1)

In [ ]:
# Cell 12 - D-038 round 1: train E011-E019 on both GPUs at once (one process per T4)
# Expected time: up to about 9.5 h after Cell 1 (deadline rule, D-038 point 3); runs that cannot finish
# by then are NOT started and are listed as "not run (time)".
# Queue order = priority (D-038 point 3): seed 0 of every size first, longest (l) started at once,
# then seed 1, then seed 2. Each run: up to 60 epochs, stops after 10 epochs without a val mAP50-95 gain.
# The deadline estimate assumes all 60 epochs (worst case), so no started run is cut off.
# Every finished run is zipped at once -> /kaggle/working/<id>_package.zip (+ sha256 in packages_sha256.txt).
# A status line per running run is printed every 10 min. The test splits are never used.

import csv, glob, json, os, re, signal, subprocess, sys, threading, time

REPO = "/tmp/uwct_repo"
RAW = "/tmp/uwct_raw"
DATASET = "/tmp/uwct_yolo/lobster_mixed_v4"
W = "/kaggle/working"
RUNS = f"{W}/uwct_runs"           # kept when the notebook version is saved
WEIGHTS = f"{W}/uwct_weights"     # pretrained checkpoint downloads
LOGS = f"{W}/uwct_logs"
CFG = f"{REPO}/configs/experiments"
os.makedirs(RUNS, exist_ok=True); os.makedirs(WEIGHTS, exist_ok=True)

ENV = dict(os.environ,
           UWCT_RAW_ROOT=f"{RAW}/biology",
           UWCT_HARSH_MODEL_A_ROOT=f"{RAW}/harsh_model_a_2026-11",
           UWCT_HARSH_V1_EXTRA_ROOT=f"{RAW}/harsh_v1_extra_2026-11",
           UWCT_HARSH_V1_COLOUR_ROOT=f"{RAW}/harsh_v1_colour_2026-11")

N_GPUS = len([l for l in subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines()
              if l.startswith("GPU ")])
PRINT_LOCK = threading.Lock()

def say(*a):
    with PRINT_LOCK:
        print(*a, flush=True)

def launch(config_name, gpu, log_path, extra=()):
    """One train_yolo.py process that sees only GPU `gpu` (it becomes its cuda:0)."""
    env = dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu))
    cmd = [sys.executable, f"{REPO}/scripts/train_yolo.py", "--config", f"{CFG}/{config_name}",
           "--runs-root", RUNS, "--weights-dir", WEIGHTS, "--dataset", DATASET, *extra]
    log = open(log_path, "w")
    return subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, text=True, env=env,
                            start_new_session=True), log

def peak_gpu_gb(log_path):
    vals = [float(x) for x in re.findall(r"\s(\d+(?:\.\d+)?)G\s", open(log_path, errors="replace").read())]
    return max(vals) if vals else float("nan")

def rows_of(results_csv):
    if not os.path.exists(results_csv):
        return []
    with open(results_csv) as fh:
        return [{k.strip(): v for k, v in r.items()} for r in csv.DictReader(fh)]

QUEUE = [("E011", "E011_yolos_ep60_seed0.yaml", "s"), ("E013", "E013_yolol_ep60_seed0.yaml", "l"),
         ("E012", "E012_yolom_ep60_seed0.yaml", "m"), ("E016", "E016_yolos_ep60_seed1.yaml", "s"),
         ("E014", "E014_yolom_ep60_seed1.yaml", "m"), ("E015", "E015_yolol_ep60_seed1.yaml", "l"),
         ("E019", "E019_yolos_ep60_seed2.yaml", "s"), ("E017", "E017_yolom_ep60_seed2.yaml", "m"),
         ("E018", "E018_yolol_ep60_seed2.yaml", "l")]
EPOCHS = 60               # ceiling (patience 10 usually stops earlier); used for the worst-case estimate
DEADLINE_H = 9.5          # D-038: a run starts only if its estimate ends before this (hours after Cell 1)
HARD_STOP_H = 11.3        # safety: Kaggle ends a background version at 12 h and would lose the outputs
OVERHEAD_S = 600          # preflight (view hashes) + finalize (val of best.pt) per run
DEFAULT_EPOCH_S = {"s": 180, "m": 300, "l": 450}   # used until a run of that size has measured epochs
T0 = float(open(f"{LOGS}/t0.txt").read())
elapsed_s = lambda: time.time() - T0

pending = list(QUEUE)
status = {exp: {"model": size, "state": "queued"} for exp, _, size in QUEUE}
LOCK = threading.Lock()

def save_status():
    with open(f"{LOGS}/queue_status.json", "w") as fh:
        json.dump({"elapsed_h": round(elapsed_s() / 3600, 2), "runs": status}, fh, indent=1)

def epoch_s(size):
    """Slowest measured seconds per epoch of any run of this size so far, else the default."""
    measured = []
    for exp, _, s in QUEUE:
        rows = rows_of(f"{RUNS}/{exp}/train/results.csv") if s == size else []
        if len(rows) >= 2:
            measured.append(float(rows[-1]["time"]) / len(rows))
    return max(measured) if measured else DEFAULT_EPOCH_S[size]

def package(exp):
    import hashlib, zipfile
    pkg = f"{W}/{exp}_package.zip"
    with zipfile.ZipFile(pkg, "w", zipfile.ZIP_DEFLATED) as z:
        for d, _, files in os.walk(f"{RUNS}/{exp}"):
            for n in sorted(files):
                p = os.path.join(d, n); z.write(p, os.path.relpath(p, W))
        for p in (f"{LOGS}/train_{exp}.log", f"{LOGS}/smoke_summary.json", f"{LOGS}/code_version.txt"):
            if os.path.exists(p):
                z.write(p, os.path.relpath(p, W))
    h = hashlib.sha256()
    with open(pkg, "rb") as fh:
        for b in iter(lambda: fh.read(1 << 20), b""):
            h.update(b)
    with LOCK, open(f"{W}/packages_sha256.txt", "a") as fh:
        fh.write(f"{h.hexdigest()}  {exp}_package.zip\n")
    return h.hexdigest(), os.path.getsize(pkg)

def lane(gpu):
    while True:
        with LOCK:
            job = None
            while pending:
                exp, cfg, size = pending.pop(0)
                est = epoch_s(size) * EPOCHS + OVERHEAD_S
                if elapsed_s() + est <= DEADLINE_H * 3600:
                    job = (exp, cfg, size, est)
                    status[exp].update(state="running", gpu=gpu, started_h=round(elapsed_s() / 3600, 2),
                                       estimate_h=round(est / 3600, 2))
                    break
                status[exp].update(state="not run (time)", estimate_h=round(est / 3600, 2),
                                   at_h=round(elapsed_s() / 3600, 2))
                say(f"SKIP {exp}: estimated {est / 3600:.2f} h would end after the {DEADLINE_H} h deadline")
            save_status()
        if job is None:
            return
        exp, cfg, size, est = job
        say(f"START {exp} (yolo26{size}) on GPU {gpu} at {elapsed_s() / 3600:.2f} h, estimate {est / 3600:.2f} h")
        t0 = time.time()
        proc, log = launch(cfg, gpu, f"{LOGS}/train_{exp}.log")
        while proc.poll() is None:
            if elapsed_s() > HARD_STOP_H * 3600:
                os.killpg(proc.pid, signal.SIGTERM); proc.wait()
                say(f"HARD STOP {exp}: killed at {elapsed_s() / 3600:.2f} h (last.pt kept, no run record)")
                break
            time.sleep(30)
        log.close()
        hours = round((time.time() - t0) / 3600, 2)
        rec_path = f"{RUNS}/{exp}/run_record.json"
        if proc.returncode == 0 and os.path.exists(rec_path):
            vm = json.load(open(rec_path))["val_metrics"]
            sha, size_b = package(exp)
            with LOCK:
                status[exp].update(state="finished", hours=hours, val=vm, package_sha256=sha, package_bytes=size_b)
                save_status()
            say(f"DONE  {exp} (yolo26{size}) {hours:.2f} h  VAL P {vm['precision']:.4f} R {vm['recall']:.4f} "
                f"mAP50 {vm['map50']:.4f} mAP50-95 {vm['map50_95']:.4f}  package {size_b:,} B sha256 {sha[:12]}...")
        else:
            tail = open(f"{LOGS}/train_{exp}.log", errors="replace").read().splitlines()[-15:]
            with LOCK:
                status[exp].update(state=f"failed (exit {proc.returncode})", hours=hours, log_tail=tail)
                save_status()
            say(f"FAILED {exp} exit {proc.returncode} after {hours:.2f} h. Last log lines:\n  " + "\n  ".join(tail))

print(f"{N_GPUS} GPU(s); queue of {len(QUEUE)} runs; started at {elapsed_s() / 3600:.2f} h after Cell 1; "
      f"deadline {DEADLINE_H} h; {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())}")
threads = [threading.Thread(target=lane, args=(g,), daemon=True) for g in range(max(1, min(N_GPUS, 2)))]
[t.start() for t in threads]
while any(t.is_alive() for t in threads):
    for t in threads:
        t.join(timeout=600 / len(threads))
    with LOCK:
        running = [e for e, s in status.items() if s["state"] == "running"]
    for exp in running:
        rows = rows_of(f"{RUNS}/{exp}/train/results.csv")
        if rows:
            r = rows[-1]
            say(f"  .. {exp}: epoch {len(rows)}/{EPOCHS}  mAP50-95 {float(r['metrics/mAP50-95(B)']):.3f}  "
                f"mAP50 {float(r['metrics/mAP50(B)']):.3f}  "
                f"{float(r['time']) / len(rows):.0f} s/epoch  (elapsed {elapsed_s() / 3600:.2f} h)")
save_status()
print("\nQUEUE FINISHED")
for exp, s in status.items():
    print(f"  {exp} yolo26{s['model']}: {s['state']}" + (f"  mAP50-95 {s['val']['map50_95']:.4f}  mAP50 {s['val']['map50']:.4f}" if "val" in s else ""))

In [ ]:
# Cell 13 - round-1 summary (val only, D-038 point 4: selection metric val mAP50-95) + logs package
# Expected time: under 1 min. Claude recomputes this locally from the run records before any decision.
import json, os, statistics, zipfile

W = "/kaggle/working"
st = json.load(open(f"{W}/uwct_logs/queue_status.json"))["runs"]
by = {}
for exp, s in st.items():
    if s["state"] == "finished":
        by.setdefault(s["model"], []).append((exp, s["val"]))
print("model | runs | val mAP50-95 mean (min-max) | mAP50 mean | P mean | R mean")
for size in ("s", "m", "l"):
    v = by.get(size, [])
    if not v:
        print(f"yolo26{size} | 0 | -"); continue
    m95 = [x["map50_95"] for _, x in v]
    mean = lambda k: statistics.mean(x[k] for _, x in v)
    print(f"yolo26{size} | {len(v)} ({', '.join(e for e, _ in v)}) | {statistics.mean(m95):.4f} "
          f"({min(m95):.4f}-{max(m95):.4f}) | {mean('map50'):.4f} | {mean('precision'):.4f} | {mean('recall'):.4f}")
print("reference (mAP50-95 / mAP50): E007 (yolo26s, 100 ep) 0.5993 / 0.8423 | E001 (v1) 0.6115 / 0.8527")

with zipfile.ZipFile(f"{W}/logs_package.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for d, _, files in os.walk(f"{W}/uwct_logs"):
        for n in sorted(files):
            p = os.path.join(d, n); z.write(p, os.path.relpath(p, W))
    if os.path.exists(f"{W}/packages_sha256.txt"):
        z.write(f"{W}/packages_sha256.txt", "packages_sha256.txt")
print("\n" + open(f"{W}/packages_sha256.txt").read() if os.path.exists(f"{W}/packages_sha256.txt") else "no packages")